# IS403 — Walmart M5: Environment and Data Check

Validate the three processed Level 1 files before forecasting. The expected split is
1,885 training days, 28 validation days, and 28 test days. This notebook checks
data integrity and records the environment; it does not train models or report
test-target statistics or forecast scores.

On Kaggle, attach the dataset containing `train.csv`, `val.csv`, and `test.csv`,
then run all cells with CPU. Input discovery is automatic when exactly one
complete dataset is attached. Internet access and GPU are unnecessary.

In [ ]:
from pathlib import Path
from importlib import metadata
from datetime import datetime, timezone
import hashlib
import json
import platform

import numpy as np
import pandas as pd

# Optional overrides. Leave both as None for the standard Kaggle workflow.
DATA_DIR = None
OUTPUT_DIR = None

## 1. Locate the processed dataset

Keep the three original filenames in the same folder. When multiple complete
datasets are available, set `DATA_DIR` to the intended folder above.

In [ ]:
FILENAMES = ('train.csv', 'val.csv', 'test.csv')


def find_data_directory(override=None):
    if override is not None:
        selected = Path(override).expanduser().resolve()
        if not all((selected / name).is_file() for name in FILENAMES):
            raise FileNotFoundError(f'Expected all three CSV files in: {selected}')
        return selected

    candidates = set()
    local_candidates = [Path.cwd(), Path.cwd() / 'dataset',
                        Path.cwd() / '02_Data' / 'processed']
    for directory in local_candidates:
        if all((directory / name).is_file() for name in FILENAMES):
            candidates.add(directory.resolve())

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        for train_path in kaggle_input.rglob('train.csv'):
            if all((train_path.parent / name).is_file() for name in FILENAMES):
                candidates.add(train_path.parent.resolve())

    if not candidates:
        raise FileNotFoundError(
            'No complete dataset found. Attach a Kaggle dataset with train.csv, '
            'val.csv, and test.csv in one folder, or set DATA_DIR explicitly.'
        )
    if len(candidates) > 1:
        options = '\n'.join(str(path) for path in sorted(candidates))
        raise ValueError(f'Multiple datasets found. Set DATA_DIR explicitly:\n{options}')
    return candidates.pop()


data_dir = find_data_directory(DATA_DIR)
output_dir = Path(OUTPUT_DIR) if OUTPUT_DIR is not None else (
    Path('/kaggle/working') if Path('/kaggle/working').is_dir()
    else Path.cwd() / 'environment_check_output'
)
print(f'Data: {data_dir}')
print(f'Output: {output_dir.resolve()}')

## 2. Validate schema, dates, and the audited data values

The reference fingerprints describe table values, so differences in CSV line
endings do not change the result. Literal `None` event labels remain strings.
The test file is checked only for data integrity; it is not used for model
selection or exploratory target statistics.

In [ ]:
EXPECTED_COLUMNS = ['d', 'date', 'total_sales', 'weekday', 'wday_iso', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']
NUMERIC_COLUMNS = ['total_sales', 'wday_iso', 'month', 'year', 'snap_CA', 'snap_TX', 'snap_WI']
EVENT_COLUMNS = ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']
REFERENCE_SHA256 = {'train': 'dcb9407590b16f6b975a1647c1c20f3cb932bc78dd75c1637f94d313c599d271', 'val': 'fbee2683fca3c5fc731b1d81829a56aa12c71e5b70ad124a1ac01ab3754536b5', 'test': '7010e850a70a6384fa282fbacfba82c752e1238ea4a962446f264fff341b842f'}
SPLIT_DAYS = {'train': (1, 1885), 'val': (1886, 1913), 'test': (1914, 1941)}
FIRST_DATE = pd.Timestamp('2011-01-29')


def require(condition, message):
    if not condition:
        raise ValueError(message)


def table_sha256(frame):
    canonical_csv = frame[EXPECTED_COLUMNS].to_csv(index=False, lineterminator='\n')
    return hashlib.sha256(canonical_csv.encode('utf-8')).hexdigest()


def validate_split(name, directory):
    path = directory / f'{name}.csv'
    frame = pd.read_csv(path, keep_default_na=False)
    require(list(frame.columns) == EXPECTED_COLUMNS,
            f'{name}: columns or column order differ from the audited dataset.')

    first, last = SPLIT_DAYS[name]
    require(len(frame) == last - first + 1, f'{name}: incorrect row count.')
    require(frame['d'].tolist() == [f'd_{day}' for day in range(first, last + 1)],
            f'{name}: day IDs are missing, duplicated, or out of order.')
    dates = pd.to_datetime(frame['date'], format='%Y-%m-%d', errors='raise')
    expected_dates = pd.date_range(FIRST_DATE + pd.Timedelta(days=first - 1),
                                   periods=len(frame), freq='D')
    require(pd.DatetimeIndex(dates).equals(expected_dates),
            f'{name}: dates do not match the fixed chronological split.')
    require(not frame.duplicated(['d']).any() and not dates.duplicated().any(),
            f'{name}: duplicate day or date.')
    frame['date'] = dates.dt.strftime('%Y-%m-%d')

    for column in NUMERIC_COLUMNS:
        values = pd.to_numeric(frame[column], errors='raise')
        require(bool(np.isfinite(values.to_numpy()).all()),
                f'{name}: {column} contains missing or non-finite values.')
        require(bool((values % 1 == 0).all()), f'{name}: {column} must contain integers.')
        frame[column] = values.astype(np.int64)

    require(bool((frame['total_sales'] >= 0).all()), f'{name}: negative sales.')
    require(np.array_equal(frame['wday_iso'], dates.dt.dayofweek + 1),
            f'{name}: incorrect ISO weekday (Monday=1).')
    require(np.array_equal(frame['month'], dates.dt.month), f'{name}: incorrect month.')
    require(np.array_equal(frame['year'], dates.dt.year), f'{name}: incorrect year.')
    require(frame['weekday'].tolist() == dates.dt.day_name().tolist(),
            f'{name}: incorrect weekday names.')
    for column in ['snap_CA', 'snap_TX', 'snap_WI']:
        require(bool(frame[column].isin([0, 1]).all()), f'{name}: {column} is not binary.')
    for column in EVENT_COLUMNS:
        require(bool(frame[column].map(lambda value: isinstance(value, str)
                     and bool(value.strip())).all()),
                f'{name}: empty event labels; expected the literal string None.')

    fingerprint = table_sha256(frame)
    require(fingerprint == REFERENCE_SHA256[name],
            f'{name}: table values differ from the audited handoff. '
            'Use the original processed CSV; do not replace it with the 10-pair output.')
    inventory = {
        'file': path.name,
        'rows': len(frame),
        'columns': len(frame.columns),
        'first_day': frame['d'].iloc[0],
        'last_day': frame['d'].iloc[-1],
        'first_date': frame['date'].iloc[0],
        'last_date': frame['date'].iloc[-1],
        'table_sha256': fingerprint,
        'matches_audited_handoff': True,
    }
    return frame, inventory

In [ ]:
frames = {}
inventory = []
for split in SPLIT_DAYS:
    frames[split], record = validate_split(split, data_dir)
    inventory.append(record)

all_dates = pd.DatetimeIndex(pd.concat([frames[name]['date'] for name in SPLIT_DAYS],
                                       ignore_index=True))
require(all_dates.equals(pd.date_range(FIRST_DATE, periods=1941, freq='D')),
        'The combined splits must contain exactly 1,941 consecutive days.')

train_values = frames['train']['total_sales'].to_numpy(dtype=np.float64)
rmsse_scale_train = float(np.mean(np.diff(train_values) ** 2))
require(np.isfinite(rmsse_scale_train) and rmsse_scale_train > 0,
        'The training first-difference mean square must be positive for RMSSE.')

print(pd.DataFrame(inventory)[['file', 'rows', 'columns', 'first_date', 'last_date',
                              'matches_audited_handoff']].to_string(index=False))
print('PASS: all three files match the audited Level 1 handoff.')

## 3. Record the environment and save the result

Package availability is recorded without installing or upgrading anything.
Only NumPy and pandas are required for this check. Dependencies for statistical,
machine-learning, and deep-learning experiments will be checked when those
notebooks are introduced.

In [ ]:
packages = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'statsmodels',
            'xgboost', 'torch', 'tensorflow']
versions = {}
for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = None

report = {
    'status': 'PASS',
    'checked_at_utc': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(),
    'platform': platform.platform(),
    'packages': versions,
    'data_directory': str(data_dir),
    'forecast_horizon': 28,
    'target': 'total_sales',
    'aggregation': 'M5 Level 1: sum of 30,490 item-store series',
    'splits': inventory,
    'rmsse_scale_original_train': rmsse_scale_train,
    'rmsse_scale_definition': 'mean squared first differences of d_1 through d_1885',
    'model_training_performed': False,
    'test_target_statistics_reported': False,
}
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / 'environment_check.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n',
                       encoding='utf-8')
print('ENVIRONMENT CHECK PASSED')
print(f'Saved: {report_path.resolve()}')